# Bybit Intra Arb01 - Latest 48h Local Snapshot

Read the latest local 48 hour parquet snapshot for `bybit-intra-arb01`. The data is expected under `data/intra_order_export_backfill/` and is loaded directly from local parquet files, without calling the persist read server.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'data' / 'intra_order_export_backfill').exists():
    REPO_ROOT = Path('/home/ubuntu/crypto_mkt/mkt_signal')

SOURCE_ID = 'bybit-intra-arb01'
EXPORT_ROOT = REPO_ROOT / 'data' / 'intra_order_export_backfill'
RUN_GLOB = 'bybit_intra_arb01_48h_*'

source_dirs = sorted(EXPORT_ROOT.glob(f'{RUN_GLOB}/{SOURCE_ID}'))
if not source_dirs:
    raise FileNotFoundError(f'No local 48h snapshot found under {EXPORT_ROOT}')

DATA_DIR = source_dirs[-1]
UNIFORM_PATH = DATA_DIR / 'uniform_orders.parquet'
ORDER_UNMATCHED_PATH = DATA_DIR / 'order_updates_unmatched.parquet'
TRADE_UNMATCHED_PATH = DATA_DIR / 'trade_updates_unmatched.parquet'

print('data dir              :', DATA_DIR)
print('uniform_orders        :', UNIFORM_PATH)
print('order_updates_unmatched:', ORDER_UNMATCHED_PATH)
print('trade_updates_unmatched:', TRADE_UNMATCHED_PATH)

## Load Local Parquet

In [ ]:
uniform_orders_48h = pd.read_parquet(UNIFORM_PATH)
order_updates_unmatched_48h = pd.read_parquet(ORDER_UNMATCHED_PATH)
trade_updates_unmatched_48h = pd.read_parquet(TRADE_UNMATCHED_PATH)

if 'ts_us' in uniform_orders_48h.columns:
    uniform_orders_48h['ts'] = pd.to_datetime(uniform_orders_48h['ts_us'], unit='us', utc=True)
    uniform_orders_48h = uniform_orders_48h.sort_values('ts_us').reset_index(drop=True)

if 'ts_us' in order_updates_unmatched_48h.columns:
    order_updates_unmatched_48h['ts'] = pd.to_datetime(order_updates_unmatched_48h['ts_us'], unit='us', utc=True)
    order_updates_unmatched_48h = order_updates_unmatched_48h.sort_values('ts_us').reset_index(drop=True)

if 'ts_us' in trade_updates_unmatched_48h.columns:
    trade_updates_unmatched_48h['ts'] = pd.to_datetime(trade_updates_unmatched_48h['ts_us'], unit='us', utc=True)
    trade_updates_unmatched_48h = trade_updates_unmatched_48h.sort_values('ts_us').reset_index(drop=True)

summary = pd.DataFrame([
    {'table': 'uniform_orders', 'rows': len(uniform_orders_48h), 'columns': len(uniform_orders_48h.columns)},
    {'table': 'order_updates_unmatched', 'rows': len(order_updates_unmatched_48h), 'columns': len(order_updates_unmatched_48h.columns)},
    {'table': 'trade_updates_unmatched', 'rows': len(trade_updates_unmatched_48h), 'columns': len(trade_updates_unmatched_48h.columns)},
])
summary

## Uniform Orders Window

In [ ]:
if uniform_orders_48h.empty:
    print('uniform_orders_48h is empty')
else:
    print('first ts:', uniform_orders_48h['ts'].iloc[0] if 'ts' in uniform_orders_48h.columns else 'n/a')
    print('last  ts:', uniform_orders_48h['ts'].iloc[-1] if 'ts' in uniform_orders_48h.columns else 'n/a')
    print('venues  :', uniform_orders_48h['trading_venue'].value_counts().to_dict() if 'trading_venue' in uniform_orders_48h.columns else 'n/a')

## Preview

In [ ]:
uniform_orders_48h.head(20)

In [ ]:
uniform_orders_48h.tail(20)

## Quick Checks

In [ ]:
if uniform_orders_48h.empty:
    pd.DataFrame(columns=['status', 'orders'])
else:
    uniform_orders_48h.groupby('status', dropna=False).size().reset_index(name='orders').sort_values('orders', ascending=False)

In [ ]:
if uniform_orders_48h.empty:
    pd.DataFrame(columns=['symbol', 'orders'])
else:
    uniform_orders_48h.groupby('symbol', dropna=False).size().reset_index(name='orders').sort_values('orders', ascending=False).head(30)

## Full DataFrames

In [ ]:
uniform_orders_48h

In [ ]:
order_updates_unmatched_48h

In [ ]:
trade_updates_unmatched_48h